# Capítulo 17 · Laboratório 17: a grade de nove (versão didática)
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap17.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. **Atenção:** esta é uma versão didática da grade de nove de Ralph Acampora, com regras públicas (13 semanas e faixa de ±3%). Não são os parâmetros do ARGON. O laboratório monta a grade do Ibovespa, mede o fôlego da grade, testa o alerta de "índice na máxima com muito vermelho" e testa os marcadores de movimento.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap17.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=0: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ACOES = ["ABEV3", "ALOS3", "ASAI3", "AZZA3", "B3SA3", "BBAS3", "BBDC4", "BBSE3", "BEEF3", "BPAC11", "BRAV3",
         "MBRF3", "CMIG4", "CMIN3", "CPFE3", "CSAN3", "CSNA3", "CXSE3", "CYRE3", "DIRR3", "EGIE3",
         "AXIA3", "EMBJ3", "ENEV3", "ENGI11", "EQTL3", "FLRY3", "GGBR4", "HAPV3", "HYPE3", "IGTI11", "IRBR3",
         "ISAE4", "ITUB4", "KLBN11", "LREN3", "MGLU3", "MOTV3", "MRVE3", "MULT3", "NATU3", "PCAR3", "PETR4",
         "PRIO3", "PSSA3", "RADL3", "RAIL3", "RAIZ4", "RDOR3", "RENT3", "SANB11", "SBSP3", "SLCE3", "SMFT3",
         "SUZB3", "TAEE11", "TIMS3", "TOTS3", "UGPA3", "USIM5", "VALE3", "VBBR3", "VIVA3", "VIVT3", "WEGE3", "YDUQ3"]
COLS = [f"{t}.SA" for t in ACOES]
LIM, JANELA = 0.03, 13                  # +-3% em 13 semanas = "de lado" (regra didática)


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):
    D = pd.read_pickle(CACHE)
else:
    d = yf.download(COLS + ["^BVSP"], start="2005-01-01", end=FIM, auto_adjust=True, progress=False,
                    group_by="column")
    confere({"as ações e o Ibovespa": d})
    D = {"Close": d["Close"], "Volume": d["Volume"]}
    pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
FECH = D["Close"][COLS].where(D["Volume"][COLS] > 0)
FECH["^BVSP"] = D["Close"]["^BVSP"]
FECH = FECH.dropna(how="all")
SEMANAL = FECH.resample("W-FRI").last()
IBOV = SEMANAL["^BVSP"]
print(f"Parte 1. {FECH[COLS].notna().any().sum()} ações do Ibovespa e o índice desde {FECH.index[0]:%Y};"
      f" último pregão {FECH.index[-1]:%d/%m/%Y}.")

## Parte 2: a grade de hoje

In [ ]:
RS = SEMANAL[COLS].div(IBOV, axis=0)                              # linha de força relativa
PRECO = SEMANAL[COLS].pct_change(JANELA, fill_method=None)
RELATIVO = RS.pct_change(JANELA, fill_method=None)
LINHA = np.select([PRECO > LIM, PRECO < -LIM], [2, 0], 1)          # 2 = preço sobe, 1 = de lado, 0 = cai
COLUNA = np.select([RELATIVO > LIM, RELATIVO < -LIM], [2, 0], 1)   # 2 = bate o índice, 1 = empata, 0 = perde
CASA = pd.DataFrame(LINHA * 3 + COLUNA + 1, index=PRECO.index, columns=COLS).where(PRECO.notna() & RELATIVO.notna())
HOJE, ANTES = CASA.iloc[-1].dropna(), CASA.iloc[-2]
GRADE = {}
print(f"\nParte 2. Grade de nove didática do Ibovespa em {FECH.index[-1]:%d/%m/%Y} ({len(HOJE)} ações);"
      " entre parênteses, quantas casas a ação andou na semana")
for k in (9, 8, 7, 6, 5, 4, 3, 2, 1):
    nomes = [f"{t[:-3]}({int(HOJE[t] - ANTES[t]):+d})" if pd.notna(ANTES[t]) and HOJE[t] != ANTES[t] else t[:-3]
             for t in HOJE[HOJE == k].index]
    GRADE[k] = nomes
    print(f"   casa {k}: {len(nomes):>2} | " + ", ".join(nomes[:12]) + (" ..." if len(nomes) > 12 else ""))

## Parte 3: o fôlego da grade

In [ ]:
N = CASA.notna().sum(axis=1)
OK = N >= 30
VERDE = ((CASA >= 7).sum(axis=1) / N)[OK]
VERMELHO = ((CASA <= 3).sum(axis=1) / N)[OK]
print(f"\nParte 3. Casas de cima (7 a 9) hoje: {pc(VERDE.iloc[-1])} | casas de baixo (1 a 3): {pc(VERMELHO.iloc[-1])}")
print(f"   Desde {VERDE.index[0]:%Y}: casas de baixo com mediana de {pc(VERMELHO.median())}, acima de 50% em"
      f" {pc((VERMELHO > 0.5).mean())} das semanas; casas de cima com mediana de {pc(VERDE.median())}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3.5))
(VERDE * 100).rolling(13).mean().plot(ax=ax, color="black", label="casas 7 a 9")
(VERMELHO * 100).rolling(13).mean().plot(ax=ax, color="gray", label="casas 1 a 3")
ax.legend()
ax.set_title("Fôlego da grade didática (porcentagem das ações, média de 13 semanas)")
plt.show()

## Parte 4: índice perto da máxima com muito vermelho

In [ ]:
PERTO = IBOV >= IBOV.rolling(52).max() * 0.97                      # até 3% da máxima de 52 semanas
F13, F26 = IBOV.shift(-JANELA) / IBOV - 1, IBOV.shift(-26) / IBOV - 1
BASE = pd.DataFrame({"verm": VERMELHO, "perto": PERTO, "f13": F13, "f26": F26}).dropna()
BASE = BASE[BASE["perto"].astype(bool)]
TAB_ALERTA = []
print(f"\nParte 4. Ibovespa até 3% da máxima de 52 semanas: {len(BASE)} semanas")
for corte in (0.2, 0.3, 0.4):
    a, b = BASE[BASE["verm"] > corte], BASE[BASE["verm"] <= corte]
    episodios = (a.index.to_series().diff() > pd.Timedelta(weeks=4)).sum() + (len(a) > 0)
    TAB_ALERTA.append((corte, len(a), episodios, a["f13"].mean(), a["f26"].mean(), len(b), b["f13"].mean(),
                       b["f26"].mean()))
    print(f"   mais de {pc(corte)} nas casas 1 a 3: {len(a):>3} semanas ({episodios} episódios), Ibovespa"
          f" {ps(a['f13'].mean())} em 13 semanas e {ps(a['f26'].mean())} em 26 | demais: {len(b):>3} semanas,"
          f" {ps(b['f13'].mean())} e {ps(b['f26'].mean())}")

## Parte 5: marcadores de movimento

In [ ]:
SALTO = CASA.diff()                                                # casas andadas na semana
FUT = RS.shift(-JANELA) / RS - 1                                   # força relativa nas 13 semanas seguintes
T = pd.DataFrame({"salto": SALTO.stack(), "casa": CASA.stack(), "depois": FUT.stack()}).dropna()
TAB_SALTO = []
print(f"\nParte 5. Marcadores de movimento ({nr(len(T))} observações): força relativa nas 13 semanas seguintes")
for nome, filtro in [("subiu 3 casas ou mais", T["salto"] >= 3), ("subiu 1 ou 2 casas", T["salto"].between(1, 2)),
                     ("ficou na mesma casa", T["salto"] == 0), ("caiu 1 ou 2 casas", T["salto"].between(-2, -1)),
                     ("caiu 3 casas ou mais", T["salto"] <= -3), ("entrou na casa 9", (T["casa"] == 9) & (T["salto"] > 0))]:
    x = T.loc[filtro, "depois"]
    TAB_SALTO.append((nome, len(x), x.mean(), (x > 0).mean()))
    print(f"   {nome:<22} {len(x):>6} obs. | média {ps(x.mean())} | bateu o índice em {pc((x > 0).mean())}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a célula da Parte 2. Mostre por que `RELATIVO` é a variação de 13 semanas da linha de força relativa `RS`, como `np.select` usa `LIM` para separar sobe, de lado e cai, e por que `LINHA * 3 + COLUNA + 1` põe na casa 9 a ação que sobe e bate o índice." Confira rodando a Parte 2, que imprime quantas ações estão em cada casa em 30/09/2026 (no livro, 42 das 66 ações nas casas 7 a 9 e 14 na casa 1).
2. **Adaptar:** "Numa célula nova, depois da Parte 2, baixe [AÇÃO] da B3 com `yf.download` (ticker com .SA, `end=FIM` e `auto_adjust=True`), passe para semanal com `resample('W-FRI').last()` e calcule a casa dela contra `IBOV` com as mesmas contas de `PRECO`, `RELATIVO`, `LINHA`, `COLUNA` e `CASA`. Não mude `JANELA` nem `LIM`." Antes de olhar a ação nova, confira se a Parte 2 continua com 66 ações e se a Parte 3 continua com os números do livro (casas de baixo com mediana de 31% desde 2007).

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque 13 semanas por 26 e a faixa de 3% por 5%. Quantas ações mudam de casa?
2. Na Parte 4, troque a distância da máxima de 3% para 5%. O alerta com mais de 40% no vermelho continua?
3. Na Parte 5, meça a força relativa 4 semanas depois, e não 13. Os marcadores passam a dizer alguma coisa?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.